# Yatırım personası — `persona-v1`, 3 saatlik bütçeyle

Bu notebook `mf-backend/internal/decision`'ın konuştuğu personayı eğitiyor:
kanıta dayanan, her iddiayı `[n]` ile kaynağa bağlayan, ve **kritik bilgi
eksikse tahmin yerine tek bir soru soran** yatırım analisti. Rubrik adapter'ı
şema dolduruyor; bu onun kardeşi değil, başka bir davranış.

Base **Qwen3-4B-Instruct-2507** — persona hattının runbook'u Gemma-2-2B için
yazılmıştı, ama ürünün servis ettiği base bu ve `llamacpp` tek base + çok LoRA
yüklüyor. Aynı base'i paylaşmayan bir adapter tek başına bir process ister.

## Bütçe — ve nereden çıktığı

| aşama | tahmin |
|---|---|
| kurulum + model indirme | ~12 dk |
| taban kapısı, 30 satır | ~20 dk |
| eğitim, `--max-steps 48` **ya da** `--max-minutes 115` | ≤115 dk |
| adapter ölçümü, 30 satır (taban kayıttan okunuyor) | ~15 dk |
| **toplam** | **~2,7 saat** |

Adım sayısı **tahmin edildi**, süre **garanti edildi**. İkisi aynı şey değil ve
farkı `rubric-curve` ödedi: 200 adım ölçülmüş bir s/satır'dan hesaplanmıştı, ama
satır/adım çarpanı yarısı kadar yazılmıştı, koşu 12 saatlik oturum duvarına
`exit 137` ile çarptı ve **hiç ağırlık yazılmadı**. Bu koşuda `--max-minutes`
var: süre dolarsa Trainer'dan nazikçe çıkılır, `trainer.train()` *döner*, ve
ondan sonraki her satır — `save_pretrained`, eval, `train_metrics.json` — yine
koşar.

Aritmetik:

* `measure_tokens.py`, Qwen3 tokenizer'ı, 800 satır: **ortalama 892 token, en
  uzunu 1054**. `--max-seq-len 1280` hiçbir şeyi kırpmıyor. Rubrik satırları
  1880/2439'du — persona satırı yarısı kadar, çünkü vaka metni değil kanıt
  listesi taşıyor.
* Rubrik hattında ölçülen maliyet 1880 token'lık satırda **29,1 s/satır**
  (`probe.json`, iki kez ölçüldü). Uzunluğun %47'sinde ~14 s/satır bekleniyor;
  bütçe **18 s/satır** üzerinden yazıldı, yani tahmin şaşarsa yukarı şaşsın.
* Satır/adım = `batch 1 × accum 4 × 2 kart` = **8**. İki kart varsayım değil:
  `machine_shape: NvidiaTeslaT4` iki kart veriyor ve Trainer üstüne sorulmadan
  DataParallel koyuyor.
* 115 dk × 60 / 18 s = 383 satır / 8 = **48 adım**.

Üç senaryo, üçü de adapter üretiyor: 14 s/satır çıkarsa koşu ~90 dakikada
**48 adımı tamamlar** ve cosine planı sonuna kadar iner; 18 çıkarsa tam
bütçede biter; 25 çıkarsa deadline ~34. adımda keser ve ağırlıklar yine yazılır.

`--grad-accum 4`, runbook'un 16'sı değil. Bütçe bağladığında aynı satır
geçişini daha çok optimizer adımına bölmek gerekiyor: pilot tam bu ayarla 21
adımda ölçülebilir bir fark üretti. Efektif batch 16 ile 48 adım yerine 24
adımımız olurdu.

In [ ]:
import glob, json, os, shutil, subprocess, sys
import torch

assert torch.cuda.is_available(), "GPU acik degil - Settings > Accelerator > GPU T4"
cap = torch.cuda.get_device_capability(0)
print("GPU:", torch.cuda.get_device_name(0), "sm_%d%d" % cap)
print("kart sayisi:", torch.cuda.device_count())
print("bellek: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))

# Fail here, in five seconds, rather than after an 8 GB download. A P100 is
# sm_60: Kaggle's torch build does not support it at all, and bitsandbytes needs
# sm_75 for 4-bit NF4. The Flutter run landed on one because kernel-metadata
# omitted machine_shape, and the error arrived half an hour in wearing a
# different mask.
assert cap >= (7, 5), (
    f"sm_{cap[0]}{cap[1]} yetersiz - 4-bit NF4 icin T4 (sm_75) gerekiyor. "
    "Settings > Accelerator > GPU T4 x2")

In [ ]:
# Qwen3 icin transformers >= 4.51 gerekiyor; Kaggle imaji eskiyse sessizce
# 'unknown architecture' ile duser.
!pip -q install -U "transformers>=4.51" "peft>=0.11" "bitsandbytes>=0.43" "accelerate>=0.30" datasets 2>&1 | tail -2
import transformers, peft, bitsandbytes
print("transformers", transformers.__version__, "| peft", peft.__version__, "| bnb", bitsandbytes.__version__)
# torchao kaldiriliyor, yukseltilmiyor. peft'in LoRA dispatcher'i sardigi her
# kuantize OLMAYAN Linear icin is_torchao_available() soruyor ve o fonksiyon
# uyumsuz surumde False donmek yerine ImportError firlatiyor. Kaggle imaji
# 0.10.0 tasiyor, peft ('peft>=0.11' artik 0.20'ye cozuluyor) >0.16.0 istiyor.
#
# Bu notebook'ta iki yerden birden vurur: olcum fp16 yukluyor (4-bit'te
# bitsandbytes kendi Linear4bit'ini once eslestirdigi icin dispatcher'a hic
# varilmiyor, o yuzden egitim kolu bunu gormez). rubric-curve-eval tam burada
# durdu — taban olcumu bittikten SONRA, adapter gecisinin ilk saniyesinde.
#
# Silmek find_spec'i None yapar ve kontrol False doner, ki dogru cevap odur:
# torchao nicemlemesi kullanmiyoruz. Yukseltmek torch'u da suruklerdi.
!pip -q uninstall -y torchao 2>&1 | tail -1

In [ ]:
def find_mount(slug, marker):
    '''Locate one input mount by the dataset/kernel slug in its path.

    Not by filename. A kernel attached with kernel_sources contributes the whole
    of its /kaggle/working, which includes its own copies of the data files and
    the scripts — so searching for a data file finds two mounts and picks between
    them by luck. The slug is the only thing that distinguishes them.

    Recursive on top of that, because the mount depth is not a promise: the same
    dataset has appeared directly under /kaggle/input and, on the next run, one
    level deeper under /kaggle/input/datasets.
    '''
    hits = [p for p in glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
            if slug.split("/")[-1] in p]
    assert hits, (f"'{slug}' bagli degil (aranan: {marker}). "
                  f"Kaggle > Notebook > Add Input, ve surumun islenmesi bitmis olmali.")
    return os.path.dirname(sorted(hits, key=len)[0])


for root, dirs, files in os.walk("/kaggle/input"):
    print(root, "->", sorted(files)[:6], "..." if len(files) > 6 else "")
    if root.count("/") > 6:
        dirs.clear()

In [ ]:
WORK = "/kaggle/working"
DATA = find_mount("emrahik/persona-dataset", "persona_train.jsonl")
print("veri seti:", DATA)

os.makedirs(f"{WORK}/data", exist_ok=True)
for f in os.listdir(DATA):
    dst = f"{WORK}/data/{f}" if f.endswith(".jsonl") else f"{WORK}/{f}"
    shutil.copy(f"{DATA}/{f}", dst)
os.chdir(WORK)
print(sorted(os.listdir(WORK)))
print(sorted(os.listdir(f"{WORK}/data")))

# The dataset carries the scripts, and this notebook passes flags that older
# copies do not have. A stale dataset version fails here, in a second, rather
# than two hours in — and the message says which half is behind.
def needs(script, *flags):
    h = subprocess.run([sys.executable, script, "--help"],
                       capture_output=True, text=True).stdout
    for f in flags:
        assert f in h, (
            f"{script} '{f}' bilmiyor — Kaggle dataset'i eski. "
            "peft/kaggle/push_persona.sh calistir, surumun islenmesini bekle, "
            "sonra bu kernel'i yeniden push et.")
    print(f"{script}: {', '.join(flags)} var")

needs("train_qlora_qwen.py", "--max-steps", "--save-steps", "--max-minutes")
needs("persona_eval.py", "--local", "--base-only", "--baseline", "--adapter")

# eval ile meta ayni uzunlukta olmali; persona_eval.py zaten cikiyor ama sebebi
# burada bir satirda gorunsun.
n_eval = sum(1 for _ in open("data/persona_eval.jsonl"))
n_meta = sum(1 for _ in open("data/persona_eval_meta.jsonl"))
n_train = sum(1 for _ in open("data/persona_train.jsonl"))
print(f"train {n_train} satir | eval {n_eval} / meta {n_meta}")
assert n_eval == n_meta, "eval ve meta uzunluklari farkli — ikisini birlikte yeniden uret"

## 1. Taban kapısı — eğitmeden önce, taban zaten yapıyor mu

Bu bölüm sayı üretmek için değil, **gereksiz bir eğitime girmemek** için. Rubrik
hattı bu dersi pahalıya öğrendi: ilk koşu, tabanın işi zaten yaptığı bilgisinin
üstünden eğitime girdi, çünkü kapı sonucu bir dosyaya yazıp geçiyordu ve hiçbir
hücre okumuyordu. Flutter v8'de aynı tavana eğitim bittikten sonra çarpıldı.

Personanın belgelenmiş taban kusurları şunlar, ve ölçülen şey bunlar:

| metrik | ne diyor |
|---|---|
| `citation_valid` | uydurma `[n]` atıfı var mı — **kapı bekçisi** |
| `grounded_format` | karar KARAR/SKOR biçiminde mi |
| `asked_when_thin` | kanıt inceyken tahmin yerine soruyor mu |
| `decision_match` | verdict bandı kanıtla uyuşuyor mu (en yumuşağı, en son okunur) |

Bu ölçüm aynı zamanda son tablonun **before** sütunu: `--base-only` sonucu
`out/persona_base.json`'a yazılıyor ve eğitimden sonraki koşu onu `--baseline`
ile okuyor. Tabanı iki kez koşturmak bir model yüklemesi artı tam bir üretim
geçişi demekti — bütçeye karşı gerçek para, ve kimsenin sebebini
söyleyemeyeceği biçimde birbirinden ayrılabilen iki sayı.

**Not.** Bu `--local` ölçümü adapter'ı ölçer, ürünün servis ettiği MLC
derlemesini değil. İkisi ayrıldığında doğru olan servis edilen derlemedir;
buradaki hiçbir sayı ürün iddiası olarak aktarılmamalı.

In [ ]:
LIMIT = 30           # satir/taraf. Her satir bir uretim cagrisi.
BASE = "Qwen/Qwen3-4B-Instruct-2507"

# subprocess.run, `!` degil. `!`'in cikis kodu hicbir yere gitmez: rubric-train'in
# ilk kosusunda egitim adim 0'da CUDA OOM ile oldu, hucre devam etti, ve Kaggle
# kernel'i COMPLETE kaydetti — geriye kanit olarak yalnizca bos bir dizin kaldi.
r = subprocess.run([sys.executable, "persona_eval.py",
                    "--local", "--base-only",
                    "--local-base-model", BASE,
                    "--eval", "data/persona_eval.jsonl",
                    "--meta", "data/persona_eval_meta.jsonl",
                    "--limit", str(LIMIT),
                    "--out", "out/persona_base.json"])
assert r.returncode == 0, f"taban olcumu coktu (exit {r.returncode}) — log yukarida"

base = json.load(open("out/persona_base.json"))["before"]
print("\ntaban:", json.dumps(base, indent=2))

# Kapi. Taban dordunde de neredeyse tamsa ogretilecek davranis yok ve 115
# dakika harcamanin gerekcesi de yok — bunu egitimden SONRA ogrenmek, rubrik
# hattinin iki kez yaptigi sey.
scored = {k: v for k, v in base.items() if k != "n" and v is not None}
print("\n" + " | ".join(f"{k} {v:.2f}" for k, v in scored.items()))
assert not all(v >= 0.95 for v in scored.values()), (
    "taban dort metrikte de >=0.95 — ogretilecek bir davranis yok, egitime girme. "
    "Bu bir hata degil, bir sonuc: personanin var olma sebebi bu base'de mevcut.")
print("\nkapi gecildi: tabanda kapatilacak bir acik var, egitime devam.")

## 2. Eğitim — 48 adım ya da 115 dakika, hangisi önce gelirse

`--max-seq-len 1280`: `measure_tokens.py` ölçtü, en uzun satır 1054 token, yani
kırpma yok. Daha düşük bir değer **soldan** kırpar ve kanıt listesinin başını
atar — modele görmediği `[1]`'e atıf yapmayı öğretir, üstelik gayet normal
görünen bir loss'la. Bu bir bütçe koşusu ama bütçe buradan çıkarılmıyor.

`--save-steps 10`: gözetimsizliğin sigortası. Deadline zaten nazik çıkış
sağlıyor, ama Kaggle'ın dinamik kotası oturumu kendi de kesebilir ve o hâlde
`save_pretrained`'e hiç varılmaz.

`PYTORCH_ALLOC_CONF=expandable_segments:True` boilerplate değil: rubric-train'in
ilk koşusu adım 0'da düştü, 14,56 GB'ın 2,54 GB'ı boşken 2,58 GB istendi —
Qwen3'ün 151.936'lık vocab'ının logits tensörü, ve 1,07 GB ayrılmış-ama-
kullanılmayan bellek. Fragmentasyon.

In [ ]:
OUT = "out/persona-v1"
MAX_STEPS, MAX_MINUTES, GRAD_ACCUM = 48, 115, 4

env = dict(os.environ, PYTORCH_ALLOC_CONF="expandable_segments:True",
           PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")

r = subprocess.run([sys.executable, "train_qlora_qwen.py",
                    "--train", "data/persona_train.jsonl",
                    "--eval", "data/persona_eval.jsonl",
                    "--base-model", BASE,
                    "--out-dir", OUT,
                    "--max-seq-len", "1280",
                    "--max-steps", str(MAX_STEPS),
                    "--max-minutes", str(MAX_MINUTES),
                    "--grad-accum", str(GRAD_ACCUM),
                    "--batch-size", "1",
                    "--save-steps", "10"], env=env)
assert r.returncode == 0, f"egitim coktu (exit {r.returncode}) — log yukarida"

# Cikis kodu 0 yetmiyor: agirliklarin varligi ayrica kontrol edilmeli, yoksa
# Save Version bos bir dizini kaydeder ve olcum onu okur.
assert os.path.exists(f"{OUT}/adapter_model.safetensors"), \
    f"egitim bitti ama adapter yazilmamis — {OUT} bos"

tm = json.load(open(f"{OUT}/train_metrics.json"))
print("\n" + json.dumps({k: tm[k] for k in
                         ("row_passes", "max_steps", "max_minutes",
                          "stopped_on_deadline", "grad_accum", "max_seq_len")},
                        indent=2))
print("\ntrain_runtime: %.0f s (%.1f dk)" % (tm["train"]["train_runtime"],
                                             tm["train"]["train_runtime"] / 60))
print("olculen maliyet: %.1f s/satir" %
      (tm["train"]["train_runtime"] / max(tm["row_passes"], 1)))
print("\nBu s/satir bir sonraki kosunun butcesi. 18 varsayilmisti.")

## 3. Ölçüm — adapter, taban kayıttan

Adapter tarafı ölçülüyor; taban `--baseline` ile bölüm 1'in sonucundan
okunuyor, yeniden koşturulmuyor. İki taraf da aynı oturumda, aynı kütüphane
sürümleriyle ve aynı greedy çözmeyle ölçülmüş oluyor — deltanın anlamlı olması
için gereken tam olarak bu.

**Hangi sayıya bakılacak.** `citation_valid` kapı bekçisi: düşerse script zaten
"do not ship this adapter" basar, ve biçimi düzeltip hâlâ olmayan kanıta atıf
yapan bir build hiç build olmamasından kötüdür — savunulamaz kararı kendinden
emin biçimde üretir. Sonra `asked_when_thin`: ürünün var olma sebebi olan
davranış, kanıt inceyken tahmin etmemek. `decision_match` en son ve trend
olarak okunur; savunulabilir bir karar bir bant kayabilir.

In [ ]:
r = subprocess.run([sys.executable, "persona_eval.py",
                    "--local",
                    "--local-base-model", BASE,
                    "--adapter", OUT,
                    "--baseline", "out/persona_base.json",
                    "--eval", "data/persona_eval.jsonl",
                    "--meta", "data/persona_eval_meta.jsonl",
                    "--limit", str(LIMIT),
                    "--out", "out/persona_v1.json"])
assert r.returncode == 0, f"olcum coktu (exit {r.returncode}) — log yukarida"

res = json.load(open("out/persona_v1.json"))
print("\n" + json.dumps(res, indent=2, ensure_ascii=False))

## Sonra ne oluyor

Sayılar iyiyse adapter GPU kutusuna iniyor ve
[`PERSONA_RUNBOOK.md`](https://github.com/) 5-8. adımları koşuyor: `merge_adapter.py
--base-model Qwen/Qwen3-4B-Instruct-2507`, `build_mlc.sh --name persona-v1`,
sonra `persona_eval.py` **tünel üzerinden** — `--local` değil. O ölçüm ürünün
servis ettiği derlemeyi ölçer ve yayına alma kararını o verir; buradaki ölçüm
"bu adapter davranışı öğrendi mi"yi cevaplar, "ürünün kararları iyileşti mi"yi
cevaplamaz.

**Bilinen sınır, buradaki hiçbir sayıya güvenmeden önce okunmalı.** Persona
veri seti de üretilmiş: 5 boyut × 2 fragment. Held-out setin kombinasyonları
train'de yok (%1 örtüşme), ama **kanıt cümlelerinin %93'ü train'de geçiyor**.
Yani `decision_match` kısmen "hangi cümle kaç puan" bilgisini hatırlamayı
ödüllendirebilir. `citation_valid` ve `asked_when_thin` bundan görece korunaklı,
çünkü ikisi de içeriğe değil **yapıya** bakıyor: `[n]` numaraları o satırdaki
kanıt sırasına bağlı ve her satırda karışıyor, "sor ya da karar ver" ayrımı da
kanıtın varlığına bağlı. Bu yüzden yukarıdaki sıralama tesadüf değil.